# Corpus profile

Letter lengths and distributions by year and recipient.

In [ ]:
from shared import paths
from IPython.display import display


## Load the corpus

In [ ]:
import pickle
from pathlib import Path
from shared.classes import Letter, IntertextualTrace, Annotation

path = Path(str(paths.CORPUS_PICKLE))
with path.open('rb') as f:
    annotated_letters = pickle.load(f)
print(type(annotated_letters))
print(len(annotated_letters))
print(annotated_letters[0])


## Letter lengths

In [ ]:
from pathlib import Path
import pandas as pd
import spacy

try:
    from shared.utils import extract_relevant_text
except ImportError:
    extract_relevant_text = None
OUTPUT_PATH = Path('letter_spacy_token_lengths.csv')
nlp = spacy.blank('en')
from shared.metadata import get_letter_id


def get_letter_text(letter):
    if isinstance(letter, dict):
        if letter.get('text') is not None:
            return str(letter['text'])
        if letter.get('anno_text') is not None:
            if extract_relevant_text is not None:
                return extract_relevant_text(letter['anno_text'])
            return str(letter['anno_text'])
        return ''
    if hasattr(letter, 'anno_text') and extract_relevant_text is not None:
        return extract_relevant_text(letter.anno_text)
    if hasattr(letter, 'text'):
        return '' if letter.text is None else str(letter.text)
    if hasattr(letter, 'anno_text'):
        return '' if letter.anno_text is None else str(letter.anno_text)
    return ''
records = []
for letter in annotated_letters:
    letter_id = get_letter_id(letter)
    text = get_letter_text(letter)
    doc = nlp(text)
    all_tokens = [token.text for token in doc if not token.is_space]
    word_tokens = [token.text for token in doc if not token.is_space and (not token.is_punct)]
    records.append(
        {
            'id': letter_id,
            'word_token_count': len(word_tokens),
            'all_spacy_token_count': len(all_tokens),
            'character_count': len(text),
        },
    )
df = pd.DataFrame(records)
summary = {
    'number_of_letters': len(df),
    'min_tokens': int(df['word_token_count'].min()),
    'max_tokens': int(df['word_token_count'].max()),
    'mean_tokens': round(df['word_token_count'].mean(), 2),
    'median_tokens': round(df['word_token_count'].median(), 2),
    'std_tokens': round(df['word_token_count'].std(), 2),
    'total_tokens': int(df['word_token_count'].sum()),
}
print('\nSummary based on word-like spaCy tokens')
print('---------------------------------------')
for key, value in summary.items():
    print(f'{key}: {value}')
print('\nPercentiles')
print('-----------')
print(df['word_token_count'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print('\nShortest letters')
print('----------------')
print(df.sort_values('word_token_count').head(10).to_string(index=False))
print('\nLongest letters')
print('---------------')
print(df.sort_values('word_token_count', ascending=False).head(10).to_string(index=False))
df.to_csv(OUTPUT_PATH, index=False)
print(f'\nSaved full table to: {OUTPUT_PATH}')


## Years

In [ ]:
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from shared.metadata import extract_year

records = []
for letter in annotated_letters:
    records.append(
        {
            'id': str(letter.id),
            'year_raw': getattr(letter, 'year', None),
            'year': extract_year(getattr(letter, 'year', None)),
            'recipient': getattr(letter, 'recipient', None),
            'norm_recipient': getattr(letter, 'norm_recipient', None),
            'reviewed': getattr(letter, 'reviewed', False),
            'is_negative_example': getattr(letter, 'is_negative_example', False),
            'has_reference': len(getattr(letter, 'intertextual_traces', [])) > 0,
        },
    )
df = pd.DataFrame(records)
df_valid = df[df['year'].between(1820, 1870, inclusive='both')].copy()
year_counts = df_valid.groupby('year').size().reindex(range(1820, 1871), fill_value=0).rename('n_letters').reset_index()
year_counts['rolling_3yr'] = year_counts['n_letters'].rolling(window=3, center=True, min_periods=1).mean()
print(f'Total letters in object list: {len(df):,}')
print(f'Letters with usable years between 1820 and 1870: {len(df_valid):,}')
print(f'Letters without usable year in this range: {len(df) - len(df_valid):,}')
print('\nTop 10 years by number of letters:')
print(year_counts.sort_values('n_letters', ascending=False).head(10).to_string(index=False))
events = {
    1836: 'Early literary career',
    1842: 'First American tour',
    1844: 'Italy and Switzerland',
    1846: 'Daily News',
    1850: 'Household Words',
    1858: 'Public readings',
    1859: 'All the Year Round',
    1867: 'Second American tour',
    1870: 'Death',
}
label_fs = 16
tick_fs = 14
legend_fs = 14
event_fs = 16
title_fs = 18
fig, ax = plt.subplots(figsize=(14, 8))
ax.bar(
    year_counts['year'],
    year_counts['n_letters'],
    width=0.85,
    color='0.35',
    label='Letters per year',
)
ax.plot(
    year_counts['year'],
    year_counts['rolling_3yr'],
    linewidth=2.4,
    color='0.05',
    label='Three-year rolling mean',
)
ymax = year_counts['n_letters'].max()
ax.set_ylim(0, ymax * 1.52)
for year, label in events.items():
    ax.axvline(year, color='0.76', linewidth=1, zorder=0)
    x_text = year + 0.15
    y_text = ymax * 1.45
    ax.text(
        x_text,
        y_text,
        label,
        rotation=90,
        va='top',
        ha='left',
        fontsize=event_fs,
        color='0.20',
        linespacing=0.9,
    )
ax.set_xlabel('Year', fontsize=label_fs)
ax.set_ylabel('Number of letters', fontsize=label_fs)
ax.set_xlim(1819.5, 1870.5)
ax.set_xticks(range(1820, 1871, 5))
ax.tick_params(axis='both', labelsize=tick_fs)
ax.legend(frameon=False, loc='upper left', fontsize=legend_fs)
ax.grid(axis='y', color='0.86', linewidth=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'letter_distribution_by_year.png', dpi=300, bbox_inches='tight')
plt.show()


## Recipients

In [ ]:
from pathlib import Path
import re
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

TOP_N = 20
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_PLOT = output_dir / 'letter_distribution_by_recipient.png'
from shared.metadata import get_letter_id
from shared.metadata import get_norm_recipient
from shared.metadata import prettify_recipient
from shared.metadata import wrap_label

records = []
for letter in annotated_letters:
    records.append({'id': get_letter_id(letter), 'norm_recipient': get_norm_recipient(letter)})
df = pd.DataFrame(records)
letters_total = len(df)
letters_with_recipient = int(df['norm_recipient'].notna().sum())
letters_without_recipient = int(df['norm_recipient'].isna().sum())
recipient_counts = df.dropna(subset=['norm_recipient']).groupby('norm_recipient').size().sort_values(ascending=False)
distinct_recipients = int(recipient_counts.shape[0])
top_counts = recipient_counts.head(TOP_N).reset_index()
top_counts.columns = ['norm_recipient', 'n_letters']
top_counts['recipient_display'] = top_counts['norm_recipient'].apply(prettify_recipient)
top_counts['recipient_display_wrapped'] = top_counts['recipient_display'].apply(wrap_label)
top_n_total_letters = int(top_counts['n_letters'].sum())
top_n_share = top_n_total_letters / letters_with_recipient * 100
singletons = int((recipient_counts == 1).sum())
singleton_share = singletons / distinct_recipients * 100
print(f'Total letters in object list: {letters_total:,}')
print(f'Letters with normalised recipient: {letters_with_recipient:,}')
print(f'Letters without normalised recipient: {letters_without_recipient:,}')
print(f'Distinct recipients after normalisation: {distinct_recipients:,}')
print(f'Top {TOP_N} recipients account for {top_n_total_letters:,} letters ({top_n_share:.2f}%).')
print(
    f'Recipients represented by only one letter: {singletons:,} ({singleton_share:.2f}% of all distinct recipients).',
)
print(f'\nTop {TOP_N} recipients by number of letters:')
print(top_counts[['recipient_display', 'n_letters']].to_string(index=False))
plot_df = top_counts.sort_values('n_letters', ascending=True).copy()
label_fs = 16
tick_fs = 14
value_fs = 13
title_fs = 18
fig, ax = plt.subplots(figsize=(14, 9))
bars = ax.barh(plot_df['recipient_display_wrapped'], plot_df['n_letters'], height=0.78, color='0.35')
xmax = plot_df['n_letters'].max()
ax.set_xlim(0, xmax * 1.14)
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + xmax * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f'{int(width)}',
        va='center',
        ha='left',
        fontsize=value_fs,
        color='0.20',
    )
ax.set_xlabel('Number of letters', fontsize=label_fs)
ax.set_ylabel('Recipient', fontsize=label_fs)
ax.tick_params(axis='both', labelsize=tick_fs)
ax.grid(axis='x', color='0.86', linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_PLOT, dpi=300, bbox_inches='tight')
plt.show()
print(f'\nSaved plot to: {OUTPUT_PLOT}')
